# Hashing:

In [ ]:
import hashlib, os

# string hashing
text = 'This is text that needs to be hashed 12%8*!'

hash = hashlib.sha256(text.encode())
hex_dig = hash.hexdigest()
print(f"SHA-256 Hash: {hex_dig}")

# for more security:
salt = os.urandom(16)
derived_key = hashlib.pbkdf2_hmac(hash_name='sha256', password=text.encode(), salt=salt, iterations=100000)
print(f"Salt: {salt.hex()}")
print(f"Hashed Password: {derived_key.hex()}")


In [ ]:
user_guess = 'This is text that needs to be hashed 12%8*!'
guess_hash = hashlib.sha256(user_guess.encode()).hexdigest()
print(guess_hash)

In [ ]:
import secrets 

def signup_user(plain_password):

    salt = os.urandom(16) # 16 bit salt
    hashed_password = hashlib.pbkdf2_hmac(
        hash_name='sha256',password=plain_password.encode(),salt=salt,iterations=100_000
    )
    return salt.hex(), hashed_password.hex()

def verify_login(login_guess, stored_salt_hex, stored_hash_hex):
    
    salt_bytes = bytes.fromhex(stored_salt_hex) # convert stored salt into bytes
    # use the same params
    guess_hash = hashlib.pbkdf2_hmac(
        hash_name='sha256',password=login_guess.encode(),salt=salt_bytes,iterations=100_000
        )
    # compare with existing stored value -- secrets to avoid timing attack
    return secrets.compare_digest(guess_hash.hex(), stored_hash_hex)



# example save to database
stored_salt, stored_hash = signup_user("SuperSecret123")

# right password
is_match_1 = verify_login("SuperSecret123", stored_salt, stored_hash)
print(f"Correct password attempt: {is_match_1}")  # Outputs: True

# wrong password
is_match_2 = verify_login("WrongPassword", stored_salt, stored_hash)
print(f"Wrong password attempt: {is_match_2}")    # Outputs: False


# RSA:

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding

private_key = rsa.generate_private_key(public_exponent=65537,key_size=2048)
public_key = private_key.public_key()
textb = b"This is a test message 123%()"
textf = f"This is an  f-test message 123%()"
signature = private_key.sign(textb,padding.PSS(mgf=padding.MGF1(hashes.SHA256()),salt_length=padding.PSS.MAX_LENGTH),hashes.SHA256())
print(f"Message: {textb.decode()}")
print(f"Signature (Hex): {signature.hex()[:60]}...")

try:
    # Verify using the Public Key
    public_key.verify(signature,textb,padding.PSS(mgf=padding.MGF1(hashes.SHA256()),salt_length=padding.PSS.MAX_LENGTH),hashes.SHA256())
    print("Verification Successful! The data is authentic and untampered.")
except Exception:
    print("Verification Failed! The data or key is invalid.")



## Data Tables:

In [ ]:
from cryptography.fernet import Fernet

# Generate a random secure key
secret_key = Fernet.generate_key()

# Save this key securely (e.g., environment variables)
print(f"Your Secret Key: {secret_key.decode()}")

In [ ]:
import pandas as pd
from cryptography.fernet import Fernet


cipher = Fernet(secret_key)
data = {
    'Name': ['Alice', 'Bob', 'Charlie'],
    'SSN': ['123-456-7890', '987-654-3210', '555-444-3333'],
    'Salary': ['95000', '115000', '72000']
}
df = pd.DataFrame(data)


def encrypt_val(val):
    if pd.isna(val):
        return val
    return cipher.encrypt(str(val).encode()).decode()

# 2. Encrypt the sensitive columns
df['SSN'] = df['SSN'].apply(encrypt_val)
df['Salary'] = df['Salary'].apply(encrypt_val)

df_enc = df.copy()
df.head()

In [ ]:
# Helper function to decrypt text data
def decrypt_val(val):
    if pd.isna(val):
        return val
    return cipher.decrypt(val.encode()).decode()

# Decrypt the columns back to normal
df['SSN'] = df['SSN'].apply(decrypt_val)
df['Salary'] = df['Salary'].apply(decrypt_val)

print(df)


In [ ]:
cipher.generate_key()

In [ ]:
import os
import pandas as pd
from cryptography.fernet import Fernet

# THIS NEEDS TO BE STORED CORRECTLY AND NOT HERE -- LEARN THIS NEXT WEEK 
key_string = os.getenv("MY_TABLE_KEY")

if not key_string:
    raise ValueError("Error: MY_TABLE_KEY environment variable is not set!")

cipher = Fernet(key_string.encode())

def decrypt_val(val):
    if pd.isna(val):
        return val
    return cipher.decrypt(val.encode()).decode()

df['SSN'] = df['SSN'].apply(decrypt_val)
print(df)
